In [ ]:
import pandas as pd
total_sample = pd.read_excel('lawncare_A_B_Testing.xlsx',sheet_name='total_sample')  # Load the total_sample data from the Excel file   
print(total_sample.head())
conversion_stats = total_sample.groupby('group_version')['sign_up'].value_counts(normalize=True).unstack()
conversion_stats['conversion_rate'] = conversion_stats['Y']
print(conversion_stats[['Y']])

total_sales = pd.read_excel('lawncare_A_B_Testing.xlsx',sheet_name='total_sales')  # Assuming the sales data is in a separate sheet
total_sales['net_revenue'] = total_sales['sale_price'] - total_sales['treatment_cost']
net_rev_total = total_sales.groupby('version_group')['net_revenue'].sum()
print(net_rev_total)

retention = total_sales.groupby('version_group').agg(
    s1_users=('user_id', lambda x: x[total_sales['session'] == 1].nunique()),
    s8_users=('user_id', lambda x: x[total_sales['session'] == 8].nunique())
)
retention['retention_rate'] = retention['s8_users'] / retention['s1_users']
print(retention)



# 1. Calculate Per-Session Metrics
session_stats = total_sales.groupby(['version_group', 'session']).agg(
    active_users=('user_id', 'nunique'),
    session_net_rev=('net_revenue', 'sum')
).reset_index()
# 2. Calculate Active ARPU
session_stats['active_arpu'] = session_stats['session_net_rev'] / session_stats['active_users']
# Pivot for side-by-side comparison
comparison = session_stats.pivot(index='session', columns='version_group', values='active_arpu')
print(comparison)



# Current state at end of Session 8
data = {
    'A': {'users': 12, 'margin': 65, 'total_profit': 5355, 'churn': 0.05},
    'B': {'users': 7, 'margin': 54, 'total_profit': 3146, 'churn': 0.01}
}

projection = []
for s in range(9, 25):
    for group in ['A', 'B']:
        # Apply churn to the user count
        data[group]['users'] *= (1 - data[group]['churn'])
        # Calculate session profit
        session_profit = data[group]['users'] * data[group]['margin']
        # Update cumulative total
        data[group]['total_profit'] += session_profit
        
        projection.append({
            'Session': s,
            'Group': group,
            'Active_Users': round(data[group]['users'], 2),
            'Cumulative_Profit': round(data[group]['total_profit'], 2)
        })

df_projection = pd.DataFrame(projection)
projection_end = df_projection[df_projection['Session'] == 25]
print(projection_end)
